# Page 12 gate: does a STaR-shaped specialist clear Act 2's best?

Page 8 ends with the strongest thing the toolbox could buy on this instrument: **majority vote
over 16 chains**. This notebook asks one question, on page 8's own items, and it exists to be
answered *before* [`docs/page_12_refactor.md`](../docs/page_12_refactor.md) is written, because
the answer decides which page 12 there is to write.

> Does `Qwen2.5-Math-1.5B-Instruct`, writing **one** chain, beat `Qwen2.5-3B-Instruct` writing
> sixteen and voting?

Half the parameters, a sixteenth of the answer-time compute, and the reasoning trained in
rather than sampled for. Three outcomes, and the spec is different in each:

| | result | what page 12 becomes |
|---|---|---|
| **A** | the specialist clears majority@16 | the planned page: training beats the whole toolbox |
| **B** | it beats one chain but not sixteen | a better page: the two axes are commensurable, and voting is still buying something |
| **C** | it lands below one chain | the 1.5B is the wrong instrument; the comparison moves to the 7B and the parameter story weakens to a compute story |

Writes `out/page_12_gate/`: `traces.jsonl`, `gate.json`, `manifest.json`. Its input is
page 7's `out/page_07/traces.jsonl` and page 8's `out/page_08/derived.json`.

**Most of the baseline is already built and is not rebuilt here.** Page 8 owns majority@16 and
one-chain accuracy, and this notebook reads those numbers rather than recomputing them. Two
things it does have to derive, because page 8's `derived.json` does not carry them: **per-item**
majority verdicts, which a paired test needs and page 8's 64-subset average cannot produce
(section 3), and **answer-time token spend**, which page 8 does not record at all and which is
the x axis of page 12's figure (section 4). Everything else new here is the specialist, which
no notebook in the set has ever run on these items.


**On the word STaR.** `Qwen2.5-Math` was not trained by STaR. Its report describes iterative
rejection-sampling fine-tuning guided by a reward model, then GRPO: STaR's *shape*, with a
reward model where STaR has a bare answer key. This notebook measures a checkpoint and says
nothing about how it was made. The lineage argument belongs to the spec, and the spec has to
make it precisely.


## 0. How to run this

| Mode | What it does | Hardware | Time |
|---|---|---|---|
| **`PLAN`** *(default)* | everything except the generation: pins the items, rebuilds the baselines, prints the projected cost | **none** | seconds |
| `QUICK` | the gate on 20 of the 100 items | ~4 GB GPU | a minute or two |
| `FULL` | the gate | ~4 GB GPU | minutes |

There is no `REPLAY` here, because nothing has been published to replay yet. `PLAN` is what
`REPLAY` will become once this has run once.

**`PLAN` is the default on purpose.** Opening this notebook and running every cell must not
start a GPU job on a machine that is busy.

```bash
python -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
jupyter lab
```

Run page 7 and page 8 first, or drop their published outputs into `out/page_07/` and
`out/page_08/`.


In [ ]:
# ---------------------------------------------------------------------------
# The knobs. Everything else is derived from these.
# ---------------------------------------------------------------------------
RUN_MODE = "PLAN"         # "PLAN" (no GPU), "QUICK" (20 items), "FULL" (100 items)

# The specialist under test. The 1.5B is the interesting one: it is SMALLER than the
# 3B generalist it is measured against, so a win is not a size win.
SPECIALIST = "Qwen/Qwen2.5-Math-1.5B-Instruct"

# Only if the 1.5B lands in branch C. Runs at roughly a quarter of the 1.5B's speed
# and needs 8-bit to fit 16 GB, so it is named here and not swept by default.
FALLBACK = "Qwen/Qwen2.5-Math-7B-Instruct"
FALLBACK_QUANT = "8bit"

# Matched to page 7's pile, and changing either breaks the comparison this notebook
# exists to make. 2048 is the cap page 7's chains were given; the specialist gets the
# same one or the budget is not shared.
GEN_CAP = 2048
POOL_CONFIG = "t0.30-c2048"     # page 8's main configuration: the 16 chains that vote
GREEDY_CONFIG = "t0.00-c2048"   # page 7's temperature-0 line: the generalist's one chain
MAJORITY_N = 16

# 70 of the 100 items have a pool of exactly 16, where this rule is forced. The other
# 30 were widened to 24, and a rule is needed. Page 8 averages 64 random 16-subsets,
# which gives a better POINT estimate and no per-item verdict; a paired test needs one
# verdict per item, so this notebook takes a deterministic subset and reports how far
# that lands from page 8's number rather than hiding the difference. Section 3.
MAJORITY_SUBSET_RULE = "first_16_by_sample_index"

# A second, unpaired arm using the prompt Qwen recommends for these checkpoints. The
# paired comparison uses page 8's stem and nothing else, because a prompt difference
# would confound the thing being measured; this arm exists so that "you under-prompted
# the specialist" is answered with a number instead of an argument. Section 5.
ARM_RECOMMENDED = True

QUICK_ITEMS = 20
SEED = 123                # page 7's item draw, re-derived and checked in section 2

PAGE_07_TRACES_URL = ""   # set to fetch page 7's dump instead of running it


In [ ]:
import json
import random
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import requests
import torch

import harness

from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
    load_math500_test, render_prompt, extract_final_candidate, grade_answer)

OUT_DIR = Path("out/page_12_gate")
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

PAGE_07_DIR = Path("out/page_07")
PAGE_08_DIR = Path("out/page_08")
P7_TRACES = PAGE_07_DIR / "traces.jsonl"
P8_DERIVED = PAGE_08_DIR / "derived.json"

TRACES_PATH = OUT_DIR / "traces.jsonl"
GATE_PATH = OUT_DIR / "gate.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"

mean, median, wilson = harness.mean, harness.median, harness.wilson

assert RUN_MODE in ("PLAN", "QUICK", "FULL"), RUN_MODE
print("torch", torch.__version__)
print("harness", harness.fingerprint()["version"], harness.fingerprint()["sha256"][:12])
print("mode", RUN_MODE)


## 1. Page 7's pile

The items are not chosen here. They are read off the generations page 8 scored, so the
comparison is paired at the problem level by construction rather than by a matching seed.


In [ ]:
if PAGE_07_TRACES_URL and not P7_TRACES.exists():
    P7_TRACES.parent.mkdir(parents=True, exist_ok=True)
    print(f"fetching page 7's candidates from {PAGE_07_TRACES_URL}")
    P7_TRACES.write_text(requests.get(PAGE_07_TRACES_URL, timeout=300).text, encoding="utf-8")

assert P7_TRACES.exists(), (
    f"this gate is measured against page 7's generations and found none at {P7_TRACES}. "
    f"Run page_07_frequency_voting.ipynb, or set PAGE_07_TRACES_URL in the knobs cell.")

# Last write per key wins, as page 8 does: page 7's log is append-only and a configuration
# widened from eight chains to sixteen can hold two records under one key.
_latest = {}
for _line in P7_TRACES.open(encoding="utf-8"):
    if _line.strip():
        _r = json.loads(_line)
        _latest[(_r["run_id"], _r["config_id"], _r["problem_id"], _r["sample_index"])] = _r
p7 = list(_latest.values())
P7_RUN_ID = Counter(r["run_id"] for r in p7).most_common(1)[0][0]
p7 = [r for r in p7 if r["run_id"] == P7_RUN_ID]
GENERALIST = Counter(r["model_id"] for r in p7).most_common(1)[0][0]

print(f"page 7 run   {P7_RUN_ID}")
print(f"generalist   {GENERALIST}")
print(f"{len(p7):,} candidate records after collapsing duplicate keys")
print("configurations present: " + ", ".join(
    f"{k} ({v})" for k, v in sorted(Counter(r['config_id'] for r in p7).items())))


## 2. The items, pinned twice

Once from the dump, which is what page 8 actually scored, and once by re-deriving page 7's
draw from its seed. **The two must agree**, and if they do not, the seed is not the thing that
selected these problems and every downstream comparison is unpaired.


In [ ]:
DUMP_IDS = {r["problem_id"] for r in p7}
QUESTION = {}
GROUND_TRUTH = {}
for r in p7:
    QUESTION.setdefault(r["problem_id"], r["question"])
    GROUND_TRUTH.setdefault(r["problem_id"], r["ground_truth"])
print(f"{len(DUMP_IDS)} distinct items in the dump")

# The independent re-derivation. Page 7 sorts the band by unique_id, shuffles with SEED
# and takes the first n. Skipped rather than failed if MATH-500 is not on disk, since a
# missing dataset file is not evidence about the draw.
#
# ORDER MATTERS. sha256_of hashes a sequence, and page 7's hash is over the SHUFFLED
# draw, not a sorted list. So the draw order is the canonical one and the dump can only
# be compared to it as a set.
REDERIVED = None
ITEM_IDS = sorted(DUMP_IDS)
_math500_path = DATA_DIR / "math500_test.json"
try:
    _m500 = load_math500_test(local_path=_math500_path)
    _band = sorted([r for r in _m500 if int(r["level"]) in (4, 5)], key=lambda r: r["unique_id"])
    random.Random(SEED).shuffle(_band)
    _drawn = _band[: len(DUMP_IDS)]
    REDERIVED = harness.sha256_of(r["unique_id"] for r in _drawn)
    assert {r["unique_id"] for r in _drawn} == DUMP_IDS, (
        "page 7's dump and a re-derivation of its draw name DIFFERENT items. The paired "
        "comparison this gate makes is then not paired. Check SEED, the band filter and n.")
    ITEM_IDS = [r["unique_id"] for r in _drawn]      # the canonical order
    print(f"re-derived from seed {SEED}   sha256 {REDERIVED[:12]}   identical draw")
    # Carry the full problem text across, which the dump also holds but the re-derivation
    # proves is the same row of the same dataset.
    for row in _drawn:
        QUESTION[row["unique_id"]] = row["problem"]
        GROUND_TRUTH[row["unique_id"]] = row["answer"]
except FileNotFoundError:
    print(f"{_math500_path} not present; the seed cross-check is skipped and the dump stands "
          f"alone as the item set, in sorted order")

ITEM_SET_HASH = harness.sha256_of(ITEM_IDS)
print(f"item set   sha256 {ITEM_SET_HASH[:12]}")

# The strongest available check that this is page 8's population: page 7 names its item
# set in its own run_id, so the hash has to be sitting in that string.
if REDERIVED is not None:
    assert ITEM_SET_HASH[:8] in P7_RUN_ID, (
        f"item hash {ITEM_SET_HASH[:8]} does not appear in page 7's run_id {P7_RUN_ID}. "
        f"These are not the generations page 8 scored.")
    print(f"-> {ITEM_SET_HASH[:8]} found in page 7's run_id")

ITEMS = ITEM_IDS[:QUICK_ITEMS] if RUN_MODE == "QUICK" else ITEM_IDS
print(f"\nthis run scores {len(ITEMS)} items")


## 3. What the specialist has to beat

**Page 8 owns both baselines and this notebook does not second-guess them.** Its published
numbers are loaded first and are what the page will quote.

What page 8 does not publish is a **per-item** verdict, and a paired test cannot run without
one. Its majority@16 is an average over 64 random 16-subsets per item: the better point
estimate, and one that deliberately has no single answer per problem. So the per-item verdicts
are derived here under the deterministic subset rule named in the knobs, which is forced anyway
for the 70 items whose pool is exactly 16.

That derivation is then checked against page 8's published aggregate. **The two are allowed to
differ by about a point, because the subset rule differs; a larger gap means the rule is doing
real work and the pairing needs rethinking rather than reporting.**

One chain is page 7's temperature-0 line: greedy, one attempt, deterministic, so a paired test
against it carries no sampling noise on either side. It is the cleaner of the two comparisons
for exactly that reason.


In [ ]:
# Page 8's published baselines. These are the authority; everything below derives the
# per-item verdicts they do not carry, and is then checked back against them.
P8 = json.loads(P8_DERIVED.read_text()) if P8_DERIVED.exists() else None
P8_MAJ = (P8 or {}).get("gate", {}).get("majority_at_its_best")
P8_ONE = (P8 or {}).get("gate", {}).get("one_chain_at_its_best")
if P8_MAJ:
    print(f"page 8 publishes  majority@{P8_MAJ['n']} {P8_MAJ['accuracy']:.3f} "
          f"[{P8_MAJ['ci'][0]:.2f}, {P8_MAJ['ci'][1]:.2f}]   "
          f"one chain {P8_ONE['accuracy']:.3f}   at {P8_MAJ['config_id']}")
else:
    print(f"{P8_DERIVED} not present; page 8's published numbers are unavailable and the "
          f"derivation below stands alone")

greedy = {}
pool = {}
for r in p7:
    if r["config_id"] == GREEDY_CONFIG:
        greedy.setdefault(r["problem_id"], r)
    elif r["config_id"] == POOL_CONFIG:
        pool.setdefault(r["problem_id"], []).append(r)

assert MAJORITY_SUBSET_RULE == "first_16_by_sample_index", MAJORITY_SUBSET_RULE


def majority_verdict(records, n):
    """One verdict for one item: the modal answer among the first `n` chains by
    sample_index, graded. Ties go to whichever tied answer appears earliest, which is the
    only tie-break that does not need a second signal."""
    rs = sorted(records, key=lambda r: r["sample_index"])[:n]
    votes = Counter(r["vote_key"] for r in rs if r["vote_key"] is not None)
    if not votes:
        return False, None, len(rs)
    top = max(votes.values())
    winner = next(r for r in rs if votes.get(r["vote_key"]) == top)
    return bool(winner["correct"]), winner["vote_key"], len(rs)


ONE_CHAIN = {i: bool(greedy[i]["correct"]) for i in ITEMS if i in greedy}
MAJORITY = {}
POOL_DEPTH = {}
for i in ITEMS:
    if i in pool:
        ok, _key, depth = majority_verdict(pool[i], MAJORITY_N)
        MAJORITY[i] = ok
        POOL_DEPTH[i] = depth

assert len(ONE_CHAIN) == len(ITEMS), f"{len(ITEMS) - len(ONE_CHAIN)} items have no {GREEDY_CONFIG} chain"
assert len(MAJORITY) == len(ITEMS), f"{len(ITEMS) - len(MAJORITY)} items have no {POOL_CONFIG} pool"
assert min(POOL_DEPTH.values()) == MAJORITY_N, (
    f"an item voted with only {min(POOL_DEPTH.values())} chains, not {MAJORITY_N}")

ONE_CHAIN_ACC = mean(list(ONE_CHAIN.values()))
MAJORITY_ACC = mean(list(MAJORITY.values()))

rows = [["one chain, greedy", GREEDY_CONFIG, f"{ONE_CHAIN_ACC:.3f}",
         "%.3f-%.3f" % wilson(sum(ONE_CHAIN.values()), len(ONE_CHAIN)), "this notebook"],
        [f"majority@{MAJORITY_N}", POOL_CONFIG, f"{MAJORITY_ACC:.3f}",
         "%.3f-%.3f" % wilson(sum(MAJORITY.values()), len(MAJORITY)),
         f"this notebook, {MAJORITY_SUBSET_RULE}"]]
if P8_MAJ:
    rows.append([f"majority@{P8_MAJ['n']}", P8_MAJ["config_id"], f"{P8_MAJ['accuracy']:.3f}",
                 "%.3f-%.3f" % tuple(P8_MAJ["ci"]), "page 8, 64 random subsets"])
harness.table("The two baselines", ["baseline", "configuration", "accuracy", "95% CI", "source"], rows)

if P8_MAJ:
    _gap = abs(MAJORITY_ACC - P8_MAJ["accuracy"])
    print(f"\n  rebuild differs from page 8 by {_gap:.3f} ({_gap * len(ITEMS):.0f} items)")
    if _gap > 0.03:
        print("  -> larger than the subset rule should explain. Do not pair against majority "
              "until this is understood; the one-chain comparison is unaffected.")
else:
    print(f"\n  {P8_DERIVED} not present, so page 8's published number is not cross-checked")


## 4. What the generalist spent

Page 8 records no token accounting of any kind, so this is derived here rather than read. It is
not bookkeeping: page 12's figure is drawn on page 11's axes, which makes generated tokens the
x coordinate of every point on it.


In [ ]:
GREEDY_TOKENS = {i: greedy[i]["tokens_generated"] for i in ITEMS}
MAJORITY_TOKENS = {
    i: sum(r["tokens_generated"] for r in sorted(pool[i], key=lambda r: r["sample_index"])[:MAJORITY_N])
    for i in ITEMS}

SPEND = {
    "one_chain_mean_tokens": mean(list(GREEDY_TOKENS.values())),
    "one_chain_median_tokens": median(list(GREEDY_TOKENS.values())),
    f"majority_{MAJORITY_N}_mean_tokens": mean(list(MAJORITY_TOKENS.values())),
    f"majority_{MAJORITY_N}_median_tokens": median(list(MAJORITY_TOKENS.values())),
}
harness.table(
    "Answer-time compute, per problem",
    ["method", "chains", "mean tokens", "median tokens", "accuracy"],
    [["one chain, greedy", 1, f"{SPEND['one_chain_mean_tokens']:,.0f}",
      f"{SPEND['one_chain_median_tokens']:,.0f}", f"{ONE_CHAIN_ACC:.3f}"],
     [f"majority@{MAJORITY_N}", MAJORITY_N, f"{SPEND[f'majority_{MAJORITY_N}_mean_tokens']:,.0f}",
      f"{SPEND[f'majority_{MAJORITY_N}_median_tokens']:,.0f}", f"{MAJORITY_ACC:.3f}"]],
    note=f"{GENERALIST}, cap {GEN_CAP}. The specialist's own spend is measured in section 7 "
         f"and lands in the same table.")


## 5. The two arms

**`page8`** is the paired arm and the only one the verdict is read from. It uses ch03's stem,
unchanged, because that is the stem every chain it is compared against was written under. Any
prompt improvement here would be indistinguishable from the training effect the gate is trying
to measure.

**`recommended`** is the prompt Qwen documents for these checkpoints. It is not paired-fair and
its number is a sensitivity check: it bounds how much of any shortfall in the `page8` arm is a
prompting artefact.


In [ ]:
# Qwen's documented system prompt for the Qwen2.5-Math instruct checkpoints, quoted so a
# reader can see exactly what the second arm changes.
RECOMMENDED_SYSTEM = ("Please reason step by step, and put your final answer within "
                      "\\boxed{}.")

ARMS = ["page8"] + (["recommended"] if ARM_RECOMMENDED else [])


def build_messages(arm, question):
    """The chat turns for one arm. `page8` is ch03's stem and nothing else, so the
    specialist is asked exactly what page 7's chains were asked."""
    if arm == "page8":
        return [{"role": "user", "content": render_prompt(question)}]
    return [{"role": "system", "content": RECOMMENDED_SYSTEM},
            {"role": "user", "content": question}]


for arm in ARMS:
    print(f"--- {arm} " + "-" * 60)
    print(json.dumps(build_messages(arm, "What is $1+1$?"), indent=1))


## 6. What this will cost

Projected from page 7's own greedy chains rather than from a guess. The worst case is every
chain running to the cap, which is what a math-tuned checkpoint failing to terminate on an
unfamiliar stem would look like, and it is bounded.


In [ ]:
EXPECTED_TOKENS = int(SPEND["one_chain_mean_tokens"] * len(ITEMS) * len(ARMS))
WORST_TOKENS = GEN_CAP * len(ITEMS) * len(ARMS)

print(f"  {len(ITEMS)} items x {len(ARMS)} arm(s) = {len(ITEMS) * len(ARMS)} generations")
print(f"  expected {EXPECTED_TOKENS:,} tokens   (page 7's greedy mean, "
      f"{SPEND['one_chain_mean_tokens']:,.0f} per chain)")
print(f"  worst case {WORST_TOKENS:,} tokens   (every chain runs to the {GEN_CAP} cap)")

WARMUP_TOK_S = None
if RUN_MODE == "PLAN":
    # No weights are touched in PLAN, so there is no measured rate to project from. The
    # reference point is experiment 3's wall clock, and it is stated as the estimate it is.
    for label, rate in (("~195 tok/s (1.5B bf16, derived from experiment 3)", 195),):
        print(f"\n  at {label}:")
        print(f"    expected   {EXPECTED_TOKENS / rate / 60:5.1f} min")
        print(f"    worst case {WORST_TOKENS / rate / 60:5.1f} min")
    print("\n  PLAN stops here. Set RUN_MODE to QUICK or FULL to generate.")


## 7. The sweep

One greedy chain per item per arm, under page 7's cap, with the chat template applied exactly
as page 7 applied it. Greedy is deterministic on one machine and not between them, so the
device is recorded rather than assumed.

**The cap is clipped per item from the model's own context window.** These checkpoints ship a
4,096-token context against the generalist's 32k, and a chain scored under a smaller budget
than the chains it is compared against measures the budget rather than the model. The floor
that clip reaches is printed and asserted.


In [ ]:
RUN_ID = (f"p12gate-{SPECIALIST.split('/')[-1]}-n{len(ITEMS)}-{ITEM_SET_HASH[:8]}"
          f"-cap{GEN_CAP}-{RUN_MODE.lower()}")
STORE = harness.TraceStore(TRACES_PATH, RUN_ID, ("arm", "problem_id"))
STORE.report()

CAP_FLOOR = GEN_CAP
CONTEXT = None
model_meta = {}

if RUN_MODE != "PLAN":
    device = get_device()
    DTYPE = harness.pick_dtype(device)
    loaded = harness.load_model(SPECIALIST, device=device, dtype=DTYPE)
    model, tok = loaded["model"], loaded["tokenizer"]
    model_meta = {k: v for k, v in loaded.items() if k not in ("model", "tokenizer")}
    EOS_IDS = harness.end_of_turn_ids(tok)
    CONTEXT = getattr(model.config, "max_position_embeddings", 32768)
    print(f"context {CONTEXT:,} tokens")

    @torch.inference_mode()
    def greedy_chain(messages, cap):
        """One greedy completion, written out rather than called through a helper: this
        is the generation the whole gate rests on and it should be readable here."""
        chat = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        enc = tok(chat, return_tensors="pt", add_special_tokens=False).to(model.device)
        n_prompt = enc["input_ids"].shape[1]
        out = model.generate(
            **enc, max_new_tokens=cap,
            # A checkpoint can ship sampling defaults in generation_config, so do_sample
            # alone is not enough: state every decoding parameter or the manifest records
            # a setup that was never configured.
            do_sample=False, temperature=None, top_p=None, top_k=None,
            pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id)
        gen = out[0][n_prompt:].tolist()
        truncated = len(gen) >= cap
        while gen and gen[-1] in EOS_IDS:
            gen.pop()
        text = tok.decode(gen, skip_special_tokens=True)
        pieces = harness.incremental_token_strings(gen, tok)
        assert "".join(pieces) == text
        return dict(trace=text, tokens=pieces, tokens_generated=len(gen),
                    prompt_tokens=n_prompt, truncated=truncated)

    for arm in ARMS:
        bar = harness.progress(ITEMS, desc=f"{SPECIALIST.split('/')[-1][:24]:<24} {arm:<12}",
                               unit="item")
        for pid in bar:
            if STORE.get(arm, pid) is not None:
                continue
            messages = build_messages(arm, QUESTION[pid])
            chat = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            n_prompt = len(tok(chat, add_special_tokens=False).input_ids)
            cap = max(64, min(GEN_CAP, CONTEXT - n_prompt - 32))
            CAP_FLOOR = min(CAP_FLOOR, cap)
            g = greedy_chain(messages, cap)
            got = extract_final_candidate(g["trace"])
            STORE.append({
                "arm": arm, "problem_id": pid, "model_id": SPECIALIST,
                "question": QUESTION[pid], "ground_truth": GROUND_TRUTH[pid],
                "cap": cap, "context": CONTEXT, **g,
                "extracted_answer": got,
                "correct": bool(grade_answer(got, GROUND_TRUTH[pid]))})

    harness.free_memory()
    print(f"\ncap floor {CAP_FLOOR} of {GEN_CAP}")
    assert CAP_FLOOR == GEN_CAP, (
        f"an item was generated under a {CAP_FLOOR}-token budget against page 7's {GEN_CAP}. "
        f"The specialist is then budget-starved relative to the chains it is compared "
        f"against, and the comparison measures the context window.")
else:
    print("PLAN: nothing generated")


## 8. The verdict

Two paired tests, both exact McNemar over the items where the two methods disagree. The
one-chain comparison is the clean one: greedy against greedy, one attempt each, no sampling
noise anywhere. The majority comparison carries the subset rule from section 3.


In [ ]:
records = [json.loads(l) for l in TRACES_PATH.open(encoding="utf-8")] if TRACES_PATH.exists() else []
records = [r for r in records if r.get("run_id") == RUN_ID]

# TraceStore has no write lock. Two processes on one run_id silently produce duplicate
# records and double-counted aggregates; this has happened before on this harness.
_keys = {(r["arm"], r["problem_id"]) for r in records}
assert len(records) == len(_keys), (
    f"{len(records)} records but {len(_keys)} unique keys: duplicates in the trace bank. "
    f"Deduplicate before trusting any number below.")

GATE = {"branch": None, "arms": {}, "baselines": {
    "one_chain": {"accuracy": ONE_CHAIN_ACC, "config_id": GREEDY_CONFIG},
    "majority": {"accuracy": MAJORITY_ACC, "config_id": POOL_CONFIG, "n": MAJORITY_N,
                 "subset_rule": MAJORITY_SUBSET_RULE},
    "published_majority": P8_MAJ}, "spend": SPEND}

if not records:
    print("no generations on disk for this run_id, so there is no verdict yet.")
else:
    rows = []
    for arm in ARMS:
        by_item = {r["problem_id"]: bool(r["correct"]) for r in records if r["arm"] == arm}
        if not by_item:
            continue
        toks = [r["tokens_generated"] for r in records if r["arm"] == arm]
        acc = mean(list(by_item.values()))
        lo, hi = wilson(sum(by_item.values()), len(by_item))
        a1, b1, p1 = harness.mcnemar(by_item, ONE_CHAIN)
        a2, b2, p2 = harness.mcnemar(by_item, MAJORITY)
        GATE["arms"][arm] = {
            "n": len(by_item), "accuracy": acc, "ci": [lo, hi],
            "mean_tokens": mean(toks), "median_tokens": median(toks),
            "truncated": mean([r["truncated"] for r in records if r["arm"] == arm]),
            "boxed": mean(["\\boxed" in r["trace"] for r in records if r["arm"] == arm]),
            "vs_one_chain": {"wins": a1, "losses": b1, "p": p1},
            "vs_majority": {"wins": a2, "losses": b2, "p": p2}}
        rows.append([arm, len(by_item), f"{acc:.3f}", f"{lo:.3f}-{hi:.3f}",
                     f"{mean(toks):,.0f}", f"{a1}/{b1}", f"{p1:.4f}", f"{a2}/{b2}", f"{p2:.4f}"])

    harness.table(
        f"{SPECIALIST.split('/')[-1]}, one greedy chain",
        ["arm", "n", "accuracy", "95% CI", "mean tokens",
         "vs 1 chain +/-", "p", f"vs maj@{MAJORITY_N} +/-", "p"],
        rows,
        note=f"Against {GENERALIST}. Wins/losses count only the items where the two disagree; "
             f"p is an exact two-sided McNemar over those.")

    # The branch, read off the paired arm and nothing else.
    a = GATE["arms"].get("page8")
    if a:
        beats_majority = a["accuracy"] > MAJORITY_ACC and a["vs_majority"]["p"] < 0.05
        beats_one_chain = a["accuracy"] > ONE_CHAIN_ACC and a["vs_one_chain"]["p"] < 0.05
        if beats_majority:
            GATE["branch"] = "A"
        elif beats_one_chain or a["accuracy"] > ONE_CHAIN_ACC:
            GATE["branch"] = "B"
        else:
            GATE["branch"] = "C"
        GATE["beats_majority"] = beats_majority
        GATE["beats_one_chain"] = beats_one_chain
        VERDICT = {
            "A": "the specialist clears majority@16. Page 12 is the planned page: one chain "
                 "from a smaller model past the best the toolbox could buy.",
            "B": "the specialist beats one chain but not sixteen. Page 12 becomes the better "
                 "page: the two axes are commensurable and voting is still buying something.",
            "C": "the specialist does not beat one greedy chain. The 1.5B is the wrong "
                 f"instrument; re-run against {FALLBACK} at {FALLBACK_QUANT} and expect "
                 "roughly four times the wall clock.",
        }[GATE["branch"]]
        print(f"\n  BRANCH {GATE['branch']}: {VERDICT}")
        if a["truncated"] > 0.1:
            print(f"  {a['truncated']:.0%} of chains hit the cap. A math-tuned checkpoint that "
                  f"does not terminate on this stem is measuring the stem, not the model.")
        if a["boxed"] < 0.9:
            print(f"  only {a['boxed']:.0%} of chains boxed an answer. Unboxed answers score "
                  f"badly because extraction falls back to the last number in the trace, so a "
                  f"low rate here understates the specialist.")


## 9. The record


In [ ]:
manifest = {
    "notebook": "page_12_gate_specialist.ipynb",
    "notebook_git_sha": harness.git_sha(),
    "harness": harness.fingerprint(),
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "run_mode": RUN_MODE,
    "run_id": RUN_ID,
    # repo_revision reaches the Hub, and get_device_name opens a CUDA context. PLAN
    # promises to touch neither the network nor the GPU, so both are recorded as None
    # rather than quietly making PLAN cost something.
    "specialist": {"model_id": SPECIALIST, "quant": "bf16",
                   "revision": harness.repo_revision(SPECIALIST) if RUN_MODE != "PLAN" else None,
                   **model_meta},
    "generalist": {"model_id": GENERALIST, "source": "page 7", "run_id": P7_RUN_ID},
    "items": {"dataset": "math500_l45", "n": len(ITEMS), "seed": SEED,
              "item_set_sha256": ITEM_SET_HASH, "rederived_sha256": REDERIVED,
              "ids": ITEMS},
    "decoding": {"greedy": True, "gen_cap": GEN_CAP, "cap_floor": CAP_FLOOR,
                 "context": CONTEXT, "arms": ARMS,
                 "recommended_system": RECOMMENDED_SYSTEM if ARM_RECOMMENDED else None},
    "baselines": {"pool_config": POOL_CONFIG, "greedy_config": GREEDY_CONFIG,
                  "majority_n": MAJORITY_N, "subset_rule": MAJORITY_SUBSET_RULE},
    "versions": {"torch": torch.__version__,
                 "transformers": __import__("transformers").__version__,
                 "gpu": (torch.cuda.get_device_name(0)
                         if RUN_MODE != "PLAN" and torch.cuda.is_available() else None)},
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
GATE_PATH.write_text(json.dumps(GATE, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"manifest.json  {harness.kb(MANIFEST_PATH):>8.1f} KB")
print(f"gate.json      {harness.kb(GATE_PATH):>8.1f} KB")
if TRACES_PATH.exists():
    print(f"traces.jsonl   {harness.kb(TRACES_PATH):>8.1f} KB")
print(f"\nbranch: {GATE['branch'] or 'not decided (PLAN, or nothing generated)'}")


## What this gate may and may not claim

- **Supported:** the accuracy of `Qwen2.5-Math-1.5B-Instruct`, greedy, one chain, on the 100
  MATH-500 level 4-5 problems page 7 drew, graded by ch03, at the precision the manifest
  records.
- **Supported:** whether that beats `Qwen2.5-3B-Instruct` at one greedy chain and at
  majority@16, paired at the problem level, with the subset rule section 3 names.
- **Not supported:** anything about *how* the specialist was trained. This measures a
  checkpoint. The STaR lineage is an argument the spec has to make from Qwen's report, and it
  is a claim about a family resemblance rather than about this measurement.
- **Not supported:** a general claim that specialists beat generalists. One band, one item
  draw, two checkpoints, one seed.
- **Not supported:** anything about the 7B, which this notebook names as a fallback and does
  not run.
